# Train-test split exploration

In [1]:
import json
import os
from pathlib import Path

import numpy as np

In [2]:
os.getcwd()

'c:\\Users\\Wiktoria\\Documents\\GitHub\\robust-radiotherapy-planning\\notebooks'

In [3]:
# find project root (folder that contains .git)
ROOT = Path.cwd()
while not (ROOT / ".git").exists():
    ROOT = ROOT.parent

# set working directory to root
os.chdir(ROOT)

print("Now working in:", Path.cwd())

Now working in: c:\Users\Wiktoria\Documents\GitHub\robust-radiotherapy-planning


In [4]:
# Config
SEED = 42

DATA_DIR = Path("src/data_full/CT")
SAVE_PATH = Path("src/data_full/data_dict.json")

TRAIN_FRACTION = 0.8
VAL_FRACTION = 0.2
FOLDS = 5

In [5]:
def build_patient_pairs(patient_ids, data_dir: Path):
    files = []

    for i in patient_ids:
        patient_dir = data_dir / f"Patient_{i}"
        first_fraction = patient_dir / f"Patient_{i}_fraction_1_.nii.gz"

        fraction_names = [
            f for f in sorted(patient_dir.glob("*.nii.gz")) if f.name != f"Patient_{i}_fraction_1_.nii.gz"
        ]

        files.extend(
            {
                "moving_image": first_fraction.as_posix(),
                "fixed_image": f.as_posix(),
            }
            for f in fraction_names
        )

    return files

In [6]:
# Get patient ids
patient_dirs = sorted(DATA_DIR.glob("Patient_*"))
ids = [p.name.split("_")[1] for p in patient_dirs]

np.random.seed(SEED)
np.random.shuffle(ids)

# Train / test split
train_num = int(len(ids) * TRAIN_FRACTION)
train_ids = ids[:train_num]
test_ids = ids[train_num:]

val_num = max(1, len(train_ids) // FOLDS)

data_dict = {}

In [7]:
# Cross-validation folds
for fold in range(FOLDS):
    val_ids = train_ids[fold * val_num : min((fold + 1) * val_num, len(train_ids))]
    fold_train_ids = [i for i in train_ids if i not in val_ids]

    data_dict[fold] = {
        "train": build_patient_pairs(fold_train_ids, DATA_DIR),
        "val": build_patient_pairs(val_ids, DATA_DIR),
    }

# Test set
data_dict["test"] = build_patient_pairs(test_ids, DATA_DIR)

# Save JSON
with open(SAVE_PATH, "w", encoding="utf-8") as f:
    json.dump(data_dict, f, indent=4)

# Using ready pipeline function

In [8]:
import sys

# Go to project root (folder containing src)
ROOT = Path.cwd()
while not (ROOT / "src").exists():
    ROOT = ROOT.parent

sys.path.append(str(ROOT / "src"))

print("Project root:", ROOT)

Project root: c:\Users\Wiktoria\Documents\GitHub\robust-radiotherapy-planning


In [9]:
from data_utils import create_data_split_dict

In [10]:
data_dict = create_data_split_dict(
    seed=42,
    train_fraction=0.8,
    folds=5,
    save=True,
)